In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.optim as optim

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [2]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
display(df.head())

,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteA,2019-01-01 00:00:00,22.20,0.0,4.8,0,0.0
1,siteA,2019-01-01 00:15:00,22.27,0.0,4.8,0,0.0
2,siteA,2019-01-01 00:30:00,22.35,0.0,4.8,0,0.0
3,siteA,2019-01-01 00:45:00,22.42,0.0,4.8,0,0.0
4,siteA,2019-01-01 01:00:00,22.50,0.0,4.8,0,0.0


In [3]:
df.groupby("Demand_Response_Flag")['Site'].count()

Demand_Response_Flag
-1      2262
 0    102011
 1       847
Name: Site, dtype: int64

In [4]:
def preprocess_data(df_in):
    df = df_in.copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([5, 6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2, 3]).astype(int)
    # Create hour of day categories
    df['Is_Afternoon'] = df['Hour'].isin(range(12, 18)).astype(int)
    df['Is_Evening'] = df['Hour'].isin(range(18, 24)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site','Demand_Response_Capacity_kW'], inplace=True)
    # Fix target variable (instead of -1 make it 2)
    df['Demand_Response_Flag'] = df['Demand_Response_Flag'].replace(-1, 2)
    return df

def standardize_data(X, subset=None):
    scaler = StandardScaler()
    if subset is not None:
        X_subset = X[subset]
        X_scaled_subset = scaler.fit_transform(X_subset)
        X_scaled = X.copy()
        X_scaled[subset] = X_scaled_subset
    else:
        X_scaled = scaler.fit_transform(X)
    return X_scaled, scaler

df = preprocess_data(df)

In [5]:
display(df.head(), df.tail())

,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Afternoon,Is_Evening
0,22.20,0.0,4.8,0,0,1,1,1,0,0,0,1,0,0
1,22.27,0.0,4.8,0,0,1,1,1,15,0,0,1,0,0
2,22.35,0.0,4.8,0,0,1,1,1,30,0,0,1,0,0
3,22.42,0.0,4.8,0,0,1,1,1,45,0,0,1,0,0
4,22.50,0.0,4.8,0,1,1,1,1,0,0,0,1,0,0


,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Demand_Response_Flag,Hour,Day,Month,Weekday,Minute,Is_Weekend,Is_Summer,Is_Winter,Is_Afternoon,Is_Evening
105115,20.57,0.0,56.33,0,22,31,12,6,45,1,0,1,0,1
105116,20.60,0.0,56.33,0,23,31,12,6,0,1,0,1,0,1
105117,20.75,0.0,56.33,0,23,31,12,6,15,1,0,1,0,1
105118,20.90,0.0,56.33,0,23,31,12,6,30,1,0,1,0,1
105119,21.05,0.0,56.33,0,23,31,12,6,45,1,0,1,0,1


In [6]:
# Assume the target column is 'Demand_Response_Flag' (3 classes: 0, 1, 2)
# If not, replace with the correct target column

# Prepare features and target
X = df.drop(columns=['Demand_Response_Flag'])
y = df['Demand_Response_Flag'].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=1e-2, random_state=42, stratify=y
)
# # Use all data
# X_scaled = X; y_train = y

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Define neural network
class Net(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 128)
        self.fc5 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(p=0.2)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(128)
        
    def forward(self, x):
        
        x = self.gelu(self.fc1(x))
        x = self.bn1(x)
        x = self.dropout(x)
        x = self.gelu(self.fc2(x))
        x = self.bn2(x)
        x = self.dropout(x)
        x = self.gelu(self.fc3(x))
        x = self.bn3(x)
        x = self.dropout(x)
        x = self.gelu(self.fc4(x))
        x = self.bn4(x)
        x = self.dropout(x)
        x = self.fc5(x)
        return x

input_dim = X_train.shape[1]
num_classes = 3

# create a neural net model
model = Net(input_dim, num_classes)

# Calculate class weights to handle class imbalance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
# Update criterion to use class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
batch_size = 1024
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# configure optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Training loop

epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
        # Calculate test loss
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            test_loss = criterion(test_outputs, y_test_tensor)
    if (epoch + 1) % 10 == 0:
        avg_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Test Loss: {test_loss.item():.4f}")

# Evaluate
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test_tensor).float().mean().item()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch 10/100, Loss: 0.2220, Test Loss: 0.2117
Epoch 20/100, Loss: 0.0810, Test Loss: 0.0917
Epoch 30/100, Loss: 0.0483, Test Loss: 0.0392
Epoch 40/100, Loss: 0.0312, Test Loss: 0.0389
Epoch 50/100, Loss: 0.0646, Test Loss: 0.0510
Epoch 60/100, Loss: 0.0405, Test Loss: 0.0260
Epoch 70/100, Loss: 0.0184, Test Loss: 0.0238
Epoch 80/100, Loss: 0.0145, Test Loss: 0.0430
Epoch 90/100, Loss: 0.0134, Test Loss: 0.0211
Epoch 100/100, Loss: 0.0153, Test Loss: 0.0179
Test Accuracy: 0.9895


In [7]:
from sklearn.metrics import f1_score

# Calculate F1 score for the test set
f1 = f1_score(y_test, predicted.numpy(), average='weighted')
print(f"Weighted F1 Score: {f1:.4f}")

Weighted F1 Score: 0.9905


In [8]:
joblib.dump(scaler, 'scaler.pkl')
# Save the trained model to a file
torch.save(model.state_dict(), 'trained_model.pth')

In [9]:
# Load the trained model from file
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('trained_model.pth'))
model_loaded.eval()

Net(
  (fc1): Linear(in_features=13, out_features=256, bias=True)
  (relu): ReLU()
  (gelu): GELU(approximate='none')
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=128, bias=True)
  (fc5): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [10]:
# Calculate the distribution of predicted classes
unique_pred, counts_pred = torch.unique(predicted, return_counts=True)
distribution_pred = dict(zip(unique_pred.tolist(), counts_pred.tolist()))
print("Distribution in predicted:", distribution_pred)

Distribution in predicted: {0: 1010, 1: 14, 2: 28}
